In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

In [2]:
df = pd.read_csv('./data/titanic_procesado.csv')

Hacemos una lipieza del campo Age porque contiene valores vacios y el entrenamientodel modelo falla, por tanto lo rellenaremos con valores de la media

In [ ]:
# 1. Comprobación inicial
print("Valores nulos en Age ANTES de imputar:", df['Age'].isnull().sum())

In [15]:
# 2. Imputación: Rellenar los valores NaN de 'Age' con la mediana de esa columna
df['Age'] = df['Age'].fillna(df['Age'].median())

In [ ]:
# 3. Verificación
print("Valores nulos en Age DESPUÉS de imputar:", df['Age'].isnull().sum())

In [16]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.450565,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.563505,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.483985,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.546155,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.546155,0.000,0.0,0.448347,1.0


In [17]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array

In [6]:
# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

GridSearch

Implementaremos una técnica llamada GridSearch, la cual nos permite encontrar los mejores hiperparámetros para nuestro modelo de aprendizaje automático. GridSearch explora exhaustivamente un conjunto predefinido de valores para cada hiperparámetro, entrenando y evaluando el modelo con cada combinación posible. Al final, selecciona la combinación que proporciona el mejor rendimiento según una métrica de evaluación específica, garantizando así que el modelo esté optimizado para obtener los mejores resultados posibles.

El primer paso para implementar GridSearch es crear un diccionario que contenga todos los modelos que queremos probar y los hiperparámetros que queramos probar en cada uno de estos. 

In [19]:
# 1. Comprobación inicial
print("Valores nulos en Age ANTES de imputar:", df['Age'].isnull().sum())

# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(n_neighbors=3),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    }
}

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)
    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

Valores nulos en Age ANTES de imputar: 0
Rendimiento de los modelos de clasificación
                             Modelo  Precisión
0  Clasificador K-Nearest Neighbors       0.82
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.82


Sin GridSearch

Para comprender la búsqueda en cuadrícula (Grid Search) más fácilmente, primero veamos cómo se entrena un modelo individualmente. 

Entrenemos una regresión logística:

In [20]:
# Esto ya lo tenemos importado. Lo ponemos nuevamente nada más de referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.80


In [21]:
model = LogisticRegression(
    C=0.5,                  # Valor de regularización
    penalty='l2',            # Tipo de penalización (l2 es la regularización Ridge)
    solver='lbfgs',         # Algoritmo de optimización
    max_iter=200,           # Número máximo de iteraciones
    class_weight='balanced' # Ajustar pesos de las clases
)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.78


c:\COPSIS\git_repos\python-projects\tareas\aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [22]:
# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

Rendimiento de los modelos de clasificación
                             Modelo  Precisión
0  Clasificador K-Nearest Neighbors       0.82
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.82


In [23]:
X_train[0]

array([0.        , 1.        , 0.60290844, 0.        , 0.        ,
       0.55559921, 1.        ])

In [25]:
y_train[0]

np.int64(0)

In [26]:
nuevos_datos = np.array([0,1,0.6159084,0,0,0.55547282,1]).reshape(1,-1)

In [27]:
mejor_estimador.predict(nuevos_datos)

array([0])

Guardar el modelo

Este último paso es necesario para la siguiente lección en la que pondremos nuestro modelo en producción. Usaremos un paquete de Python llamado Pickle, el cual nos permite serializar (guardar) objetos de Python en un archivo para luego poder cargarlos y utilizarlos en diferentes entornos, como una API o una aplicación web.

Pickle es particularmente útil cuando queremos guardar modelos entrenados o cualquier otro objeto complejo de Python. Al guardar el pipeline con Pickle, nos aseguramos de que todas las transformaciones de datos y el modelo en sí se conserven tal como fueron entrenados, permitiéndonos hacer predicciones consistentes con nuevos datos en producción sin necesidad de volver a aplicar las mismas transformaciones manualmente.

Si no has importado pickle, hazlo ahora

In [28]:
import pickle

Finalmente, guarda el modelo:

In [29]:
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)